# ECE1508 — Unified MoE / AAG Benchmark

**Author: Mohammad Al Dridi**

Runs every model variant through one identical pipeline and produces the
results table and figures for the report.

**Colab Pro setup — do this first:**

1. Runtime → Change runtime type → **L4 GPU** (24 GB). A100 also works and is
   faster, but burns compute units about 3x quicker. T4 works too, just slower.
2. Runtime → **enable background execution**, so the sweep survives closing
   the tab. This is the main reason Pro is worth it here.

Results are written to Google Drive after every variant, so a disconnect never
loses completed work.


## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Clone the project repo. Safe to re-run: always starts from /content.
REPO = "https://github.com/tmrcnl/ECE1508-DeepGenerativeModels.git"
BRANCH = "feat-benchmark-and-aag"

import os, shutil
%cd /content
if os.path.exists("/content/ECE1508-DeepGenerativeModels"):
    shutil.rmtree("/content/ECE1508-DeepGenerativeModels")
!git clone --branch {BRANCH} {REPO}
%cd /content/ECE1508-DeepGenerativeModels/mohammad
!pwd && ls moe_bench

In [ ]:
!pip install -q -r requirements.txt

## 1b. Mount Google Drive

Results and trained weights are written here, so they survive the runtime disconnecting. Run this before anything that writes.

In [ ]:
# Mount Drive FIRST. Every output path below lives inside it.
import os, shutil
from google.colab import drive

# An earlier `mkdir -p /content/drive/...` run before mounting leaves plain
# local folders at the mountpoint, and Colab refuses to mount over them
# ("Mountpoint must not already contain files"). Clear that case.
if os.path.exists("/content/drive") and not os.path.exists("/content/drive/MyDrive"):
    shutil.rmtree("/content/drive")
    print("cleared a stray /content/drive left by an unmounted write")

drive.mount("/content/drive")
assert os.path.isdir("/content/drive/MyDrive"), "Drive did not mount"
print("Drive mounted OK")

## 2. Smoke test

Two variants, 32 training examples, sequence length 64. Numbers are
meaningless — this only proves the pipeline runs before you spend GPU hours.

In [ ]:
!python -m moe_bench.runner --preset smoke --variants smoke --out results_smoke

## 3. The core sweep  -- OPTIONAL, SKIP IF SHORT ON TIME

Trains all five variants from scratch, including my own re-implementations of
Tamara's two MoEs. Useful as a fully controlled comparison where every model saw
identical training, but it costs ~5 hours and **section 3a supersedes it** by
scoring her and Cole's *actual* trained weights instead.

Run 3a first. Come back here only if there is time left before the deadline.

In [ ]:
OUT = "/content/drive/MyDrive/ece1508_results/core"
os.makedirs(OUT, exist_ok=True)
print("core sweep results ->", OUT)

In [ ]:
!python -m moe_bench.runner \
    --preset budget \
    --variants core \
    --out "{OUT}" \
    --save-models \
    --batch-size 8 --grad-accum 2 --epochs 1

### Where your trained weights end up

`--save-models` writes each trained variant to **`<out>/models/<variant>/`**.
Every `--out` below points inside `/content/drive/MyDrive/`, so the weights land
in **your Google Drive** and survive the runtime disconnecting.

```
MyDrive/ece1508_results/headtohead/
|-- results.json
|-- results.tex          <- paste into the report
|-- fig_*.png            <- slides
`-- models/
    `-- aag-c8/
        |-- model.safetensors   <- your trained parameters, ~1.2 GB
        |-- config.json
        `-- tokenizer.json
```

Weights are written **before** evaluation runs, so a crash in the metrics can
never cost you the training time.

Budget the space: about 1.2 GB per trained variant. The head-to-head trains one
model; the scaling sweep trains five (~6 GB). Drop `--save-models` from the
scaling cell if your Drive is tight -- that curve only needs the numbers.

## 3a. Head-to-head against the team's trained models

The comparison that goes on the slide. Three trained models scored by identical
code on an identical eval split:

| | |
|---|---|
| `tamara-moe-top1` | Tamara's `gpt2-moe-alpaca` — loaded, not trained |
| `cole-chunkmoe` | Cole's `checkpoint-1120` — loaded, not trained |
| `aag-c8` | **yours** — trained here, same recipe Cole used |
| `gpt2-dense` | untouched GPT-2, the reference line |

Upload **only `model.safetensors`** from each checkpoint folder to Drive —
`optimizer.pt` is another 2.3 GB and is not needed for evaluation.

`--preset full` matches Cole's recipe exactly (full Alpaca, sequence length 512,
batch 4 x grad-accum 8, 1 epoch) so the numbers are comparable. Budget ~3 hours
on an L4; leave background execution on.

In [ ]:
CKPT    = "/content/drive/MyDrive/ece1508_checkpoints"
OUT_H2H = "/content/drive/MyDrive/ece1508_results/headtohead"
os.makedirs(OUT_H2H, exist_ok=True)

# Verify the teammates' weights are present before burning GPU hours on a run
# that would abort at the last variant.
missing = []
for name in ("gpt2-moe-alpaca", "checkpoint-1120"):
    path = f"{CKPT}/{name}/model.safetensors"
    size = os.path.getsize(path) / 1e9 if os.path.exists(path) else 0
    print(f"{'FOUND ' if size else 'MISSING'} {name:20s} {size:.2f} GB" if size
          else f"MISSING {name}")
    if not size:
        missing.append(name)

if missing:
    raise SystemExit(
        f"Upload these to {CKPT}/<name>/model.safetensors first: {missing}
"
        "Only model.safetensors is needed -- skip optimizer.pt (2.3 GB, unused)."
    )
print("
both checkpoints present")

In [ ]:
!python -m moe_bench.runner \
    --preset full \
    --variants headtohead \
    --checkpoints "tamara-moe-top1={CKPT}/gpt2-moe-alpaca,cole-chunkmoe={CKPT}/checkpoint-1120" \
    --out "{OUT_H2H}" \
    --save-models \
    --batch-size 4 --grad-accum 8 --epochs 1

In [ ]:
!python -m moe_bench.report "{OUT_H2H}/results.json" --out "{OUT_H2H}"

from IPython.display import Image, display
for name in ["fig_perplexity", "fig_capacity", "fig_routing"]:
    display(Image(f"{OUT_H2H}/{name}.png"))

## 4. The AAG scaling curve

chunks = 1, 2, 4, 8, 16 → 16 to 3.4e19 virtual experts, at constant storage
and constant per-token compute. This is the figure that tests the proposal's
central claim.

Add `--gradient-checkpointing` if 16 chunks runs out of memory.

In [ ]:
OUT_SCALING = "/content/drive/MyDrive/ece1508_results/scaling"
os.makedirs(OUT_SCALING, exist_ok=True)

!python -m moe_bench.runner \
    --preset budget \
    --variants scaling \
    --out "{OUT_SCALING}" \
    --save-models \
    --batch-size 8 --grad-accum 2 --epochs 1

## 5. Tables and figures

Writes `results.md`, `results.tex` (paste straight into the Prism report) and
the PNGs for the slides.

In [ ]:
!python -m moe_bench.report "{OUT}/results.json" --out "{OUT}"

In [ ]:
from IPython.display import Image, display
for name in ["fig_perplexity", "fig_capacity", "fig_routing"]:
    display(Image(f"{OUT}/{name}.png"))

In [ ]:
!python -m moe_bench.report "{OUT_SCALING}/results.json" --out "{OUT_SCALING}"
display(Image(f"{OUT_SCALING}/fig_scaling.png"))

## 6. Evaluating the team's existing checkpoints

The sweep above retrains everything under one budget, which is the controlled
comparison. To also score Tamara's full-Alpaca checkpoints on the same frozen
split, point `load_checkpoint` at her Drive folder.

In [ ]:
import torch
from moe_bench import builders, data, metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = builders.load_tokenizer()
_, eval_ds = data.build_splits(tokenizer, data.PRESETS["budget"])

CHECKPOINT = "/content/drive/MyDrive/gpt2-moe-alpaca/model.safetensors"

model = builders.build("moe-top1")
info = builders.load_checkpoint(model, CHECKPOINT)
print("missing:", len(info["missing"]), " unexpected:", len(info["unexpected"]))

model.to(device)
quality = metrics.evaluate_perplexity(model, eval_ds, device)
print("perplexity:", quality["perplexity"])
print("routing:", metrics.routing_health(quality["_routing_counts"])["entropy_ratio"])